# Improving Efficiency with Recursive Designs

In Notebook 11, we saw that the Master Theorem gives us a quick way to determine the complexity of divide-and-conquer algorithms:

$$T(n) = n^d + a \cdot T\left(\frac{n}{b}\right)$$

where:
- $a$ = number of recursive calls
- $b$ = factor by which input size shrinks
- $n^d$ = cost of non-recursive work (dividing + combining)

Today we will use this as a **diagnostic tool**. Many problems have $O(n^2)$ brute-force solutions. Can divide and conquer do better? The Master Theorem tells us exactly what determines the answer: the values of $a$, $b$, and $d$.

---

## Recap: The Master Theorem

Given $T(n) = n^d + a \cdot T\left(\frac{n}{b}\right)$, compare $d$ and $\log_b a$:

1. If $d \neq \log_b a$, then $T(n) \in \Theta\left(n^{\max(d,\; \log_b a)}\right)$

2. If $d = \log_b a$, then $T(n) \in \Theta(n^d \log n)$

Think of this as three "knobs" we can turn to improve efficiency:
- **Reduce $d$**: make the non-recursive work cheaper (e.g., a smarter combine step)
- **Reduce $a$**: make fewer recursive calls
- **Increase $b$**: shrink the input faster

#PID:37
Find the complexity of these running time equations:

(1) $T(n) = 4*n^4 + 16T(n/2) \in \Theta(n^4 \log n)$

(2) $T(n) = 4*n^4 + 15T(n/2) \in \Theta(n^4)$

(3) $T(n) = 4*n^4 + 20T(n/2) \in \Theta(n^{\log_2{20}})$



---

## Problem 1: Sorting (Review)

We already know two ways to sort a list:

**Iterative approach** (selection sort, bubble sort): compare pairs of elements, swap as needed.
- Running time: $O(n^2)$

```python
def iterative_sort(L):
    for i in range(len(L)):
        for j in range(i+1, len(L)):
            if L[i] > L[j]:
                L[i], L[j] = L[j], L[i]
```
**Divide and conquer** (merge sort): split the list in half, sort each half recursively, merge the sorted halves.

```python
def merge_sort(L):
    if len(L) <= 1:
        return L
    mid = len(L) // 2
    sorted_left = merge_sort(L[:mid])
    sorted_right = merge_sort(L[mid:])
    return merge(sorted_left, sorted_right)   # merge costs O(n)
```

##### Exercise 1

Write the recurrence relation for merge sort. Then identify $a$, $b$, and $d$, and use the Master Theorem to determine its complexity.

In [ ]:
def all_ordered_pairs(L):
    for i in range(len(L)):
        for j in range(i+1, len(L)):
            print(L[i],L[j])

all_ordered_pairs(['a','z','c','h'])

#### A More General Version of the Master Theorem


Given $T(n) = c \cdot n^d + a \cdot T\left(\frac{n}{b}\right)$, compare $d$ and $\log_b a$:

1. If $d \neq \log_b a$, then $T(n) \in \Theta\left(n^{\max(d,\; \log_b a)}\right)$

2. If $d = \log_b a$, then $T(n) \in \Theta(n^d \log n)$


Often, it's not $c \cdot n^d$, but something $c_1 \cdot n^d + c_2 \cdot n^{d-1}$.  Here's the more general form (it's not technically proper, but easier to express):

$$T(n) = \Theta(n^d) + a \cdot T\left(\frac{n}{b}\right)$$

---

## Problem 2: Maximum Sublist Sum

**Problem**: Given a list of numbers, find a non-empty contiguous sublist with the largest sum. Return the sum.

**Examples**:
- `[-10, 5, -3, 4, 1, -7, 3]` → the sublist `[5, -3, 4, 1]` has the largest sum = `7`
- `[1, 2, 3]` → the entire list has the largest sum = `6`
- `[-5, -2, -1, -4]` → the sublist `[-1]` has the largest sum = `-1`

### Brute Force Approach

**Strategy**: Check every possible sublist. A sublist starts at index `i` and ends at index `j` (where `i <= j`). Compute the sum of each sublist, and keep track of the largest.

##### Exercise 2

Implement this brute-force strategy. What is its running time complexity?

In [ ]:
#PID:38
#
# Output: print the sum of each sublist
# A sublist is an interval or a pair of indices [i,j]
# To go through all sublists, we go through all pairs of indices
# 
#
def all_sums(L):   # Theta(n^2)
    for i in range(len(L)):
        cur_sum = 0
        for j in range(i, len(L)):
            cur_sum += L[j]               # constant
            print(cur_sum)
            
L = [-10, 5, -3, 4, 1, -7, 3]
# all_sums(L)


In [ ]:
#PID:39
#
#  Output: the largest sum over all intervals
#
def largest_sum(L):
    cur_max = L[0]
    for i in range(len(L)):
        cur_sum = 0
        for j in range(i, len(L)):
            cur_sum += L[j]
            if cur_sum > cur_max:
                cur_max = cur_sum
    return cur_max

L = [-10, 5, -3, 4, 1, -7, 3]
largest_sum(L)

#### Basic iterative design patterns

English: go through each item of a list, do something
```python
for x in L:
    # if you want to keep track of or update some information, you do it here!
    do_something(x)
```

English: go through each ordered pair of indices, do something.
```python
for i in range(len(L)):
    for j in range(i, len(L)):
        # if you want to keep track of or update some information, you do it here!
        do_something(i, j, L[i], L[j])
```
English: go through each unordered pair of indices, do something.
```python
for i in range(len(L)):
    for j in range(len(L)):
        if i!=j:  # if you don't want the pair L[i], L[i]
            do_something(i, j, L[i], L[j]):
```

One of the questions: how do we initialize the variable we want to keep of?

If what you keep track of is from a set of things, initialize it to one of those things.


### Divide and Conquer Approach

Can we do better with divide and conquer?

**Strategy**: Split the list in half. The maximum sublist sum must be in one of three places:
1. Entirely in the **left** half
2. Entirely in the **right** half
3. **Crossing** the middle (includes elements from both halves)

Cases 1 and 2 are solved by recursion. Case 3 requires a new idea.

In [ ]:
#PID:40
L = [-10, 5, -3, 4, 6, -5, 1, -7, 3, -2]
left = [-10, 5, -3, 4, 6]
right = [-5, 1, -7, 3, -2]

def max_sublist_sum(L):
    
    max_left = max_sublist_sum(left)
    max_right = max_sublist_sum(right)
    max_crossing = find_max_crossing_sum(L)

# Task: print out all these sublists
def find_max_crossing_sum(L):
    m = len(L)//2
    print(m)
    # all sublists that end at index m-1
    for i in range(m):
        print(L[i:m])

    # all sublists that start at index m
    for i in range(m, len(L)):
        print(L[m:i+1])

find_max_crossing_sum(L)
    

In [ ]:
#PID:41
L = [-10, 5, -3, 4, 6, -5, 1, -7, 3, -2]
left = [-10, 5, -3, 4, 6]
right = [-5, 1, -7, 3, -2]

def max_sublist_sum(L):
    
    max_left = max_sublist_sum(left)
    max_right = max_sublist_sum(right)
    max_crossing = find_max_crossing_sum(L)

# Task: print out all sums of sublists on left and right
def find_max_crossing_sum(L):
    m = len(L)//2
    # all sublists that end at index m-1
    for i in range(m):
        A = L[i:m]
        print(A, sum(A))

    print()
    # all sublists that start at index m
    for i in range(m, len(L)):
        A = L[m:i+1]
        print(A, sum(A))

find_max_crossing_sum(L)
    

In [ ]:
#PID:42
L = [-10, 5, -3, 4, 6, -5, 1, -7, 3, -2]
left = [-10, 5, -3, 4, 6]
right = [-5, 1, -7, 3, -2]

def max_sublist_sum(L):
    
    max_left = max_sublist_sum(left)
    max_right = max_sublist_sum(right)
    max_crossing = find_max_crossing_sum(L)
    return max(max_left, max_right, max_crossing)

# Task: print out all sums of on left and right in Theta(n)
def find_max_crossing_sum(L):
    m = len(L)//2
    # all sublists that end at index m-1
    sum_left = sum(L[0:m])
    max_left = sum_left
    for i in range(m):
        # print(sum_left)
        if max_left < sum_left:
            max_left = sum_left
        sum_left -= L[i]
        
    # print('max_left', max_left)
    
    # all sublists that start at index m
    sum_right = 0
    max_right = L[m]
    for i in range(m, len(L)):
        sum_right += L[i]
        if max_right < sum_right:
            max_right = sum_right
        # print(sum_right)

    # print('max_right', max_right)
    return max_left + max_right
    
find_max_crossing_sum(L)
    

In [ ]:
#PID:45

# Solve this:
def max_sublist_sum(L):             # running time: T(n) = Theta(n) + 2*T(n/2), in Theta(n log n)
    # solve the smallest cases first
    if len(L) == 1:
        return L[0]

    # solve the bigger cases--using the same strategy (recursively)
    left = L[0 : len(L)//2]                       # running time: Theta(n)
    right = L[len(L)//2 : len(L)]

    max_left = max_sublist_sum(left)               # running time: T(n/2)
    max_right = max_sublist_sum(right)             # running time: T(n/2)
    max_crossing = find_max_crossing_sum(L)        # running time: Theta(n)
    return max(max_left, max_right, max_crossing)  # running time: Theta(1)

def find_max_crossing_sum(L):
    m = len(L)//2
    # all sublists that end at index m-1
    sum_left = sum(L[0:m])
    max_left = sum_left
    for i in range(m):
        # print(sum_left)
        if max_left < sum_left:
            max_left = sum_left
        sum_left -= L[i]
        
    # print('max_left', max_left)
    
    # all sublists that start at index m
    sum_right = 0
    max_right = L[m]
    for i in range(m, len(L)):
        sum_right += L[i]
        if max_right < sum_right:
            max_right = sum_right
        # print(sum_right)

    # print('max_right', max_right)
    return max_left + max_right
    
L = [-10, 5, -3, 4, 6, -5, 1, -7, 3, -2]
max_sublist_sum(L)
    

In [ ]:
# iterative
def max_sublist_sum_iterative(L):
    cur_max = L[0]
    for i in range(len(L)):
        cur_sum = 0
        for j in range(i, len(L)):
            cur_sum += L[j]
            if cur_sum > cur_max:
                cur_max = cur_sum
    return cur_max

L = [-10, 5, -3, 4, 6, -5, 1, -7, 3, -2]
max_sublist_sum_iterative(L)

In [ ]:
import random
def gen_list(n):
    return [ random.randint(-1000,1000) for i in range(n) ]
gen_list(5)

In [ ]:
my_lists = [ gen_list(4500) for i in range(10) ]
len(my_lists)

In [ ]:
import time

start = time.perf_counter()
for L in my_lists:
    output = max_sublist_sum_iterative(L)
    print(output)
end = time.perf_counter()
print(f"Execution time: {end - start:.6f} seconds")

In [ ]:
import time

start = time.perf_counter()
for L in my_lists:
    output = max_sublist_sum(L)
    print(output)
end = time.perf_counter()
print(f"Execution time: {end - start:.6f} seconds")

#### Summary

- a simple iterative design has complexity $\Theta(n^2)$.  This has greate LOC complexity. If you want to formalize this notion, look into Kolmogorov complexity.

- a recursive design (divide in half, calculate the max sum on both sides, and also crossing). Thas more LOC (lines of code), but is much faster with $\Theta(n \log n)$.

#### Space Complexity

How much memory (RAM) a program uses.

In [ ]:
# iterative design
def max_sublist_sum_iterative(L):     # S(n) = Theta(1)
    cur_max = L[0]                    # constant
    for i in range(len(L)):           # constant
        cur_sum = 0             
        for j in range(i, len(L)):    # constant
            cur_sum += L[j]
            if cur_sum > cur_max:
                cur_max = cur_sum
    return cur_max


In [ ]:
# Solve this:
def max_sublist_sum(L):          # S(n) = Theta(n) + 2*S(n/2)
    if len(L) == 1:
        return L[0]
    left = L[0 : len(L)//2]                       # Theta(n)        
    right = L[len(L)//2 : len(L)]                 # Theta(n)
    max_left = max_sublist_sum(left)              # S(n/2)
    max_right = max_sublist_sum(right)            # S(n/2)
    max_crossing = find_max_crossing_sum(L)       # constant   
    return max(max_left, max_right, max_crossing) # constant

def find_max_crossing_sum(L):   # S(n) = constant
    m = len(L)//2
    sum_left = 0
    for i in range(m):
        sum_left += L[i]
    max_left = sum_left
    for i in range(m):
        if max_left < sum_left:
            max_left = sum_left
        sum_left -= L[i]
        
    sum_right = 0
    max_right = L[m]
    for i in range(m, len(L)):
        sum_right += L[i]
        if max_right < sum_right:
            max_right = sum_right
    return max_left + max_right
    
L = [-10, 5, -3, 4, 6, -5, 1, -7, 3, -2]
max_sublist_sum(L)
    

$$S(n) = \Theta(n) + 2 \cdot S({n \over 2})$$

$$S(n) \in \Theta(n \log n)$$

### Summary: trade-off between time and space for Max Sublist Sum problem
- Simple iterative design
  + Time: $\Theta(n^2)$
  + Space: $\Theta(1)$
- Recursive design
  + Time: $\Theta(n \log n)$
  + Space: $\Theta(n \log n)$

##### Exercise 20

For each of the following recurrences, use the Master Theorem to find the complexity. Then determine: is the bottleneck $a$ (too many recursive calls) or $d$ (combine step too expensive)? How would you try to improve it?

(a) $T(n) = n + 9T(n/3)$

(b) $T(n) = n^3 + 2T(n/2)$

(c) $T(n) = n + 2T(n/2)$

(d) $T(n) = n^2 + 4T(n/2)$